#### Error Mitigated QSVM (ZNE + REM Combined) - Lung Cancer

This notebook implements **combined error mitigation** using:
- **Zero-Noise Extrapolation (ZNE)**: Extrapolates results from multiple noise scales
- **Readout Error Mitigation (REM)**: Corrects measurement errors using calibration matrix

Focus: Comparing generalization capability of Error-Mitigated QSVM vs Classical SVM

In [ ]:
import qiskit, qiskit_aer, qiskit_machine_learning
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)
print("QML:", qiskit_machine_learning.__version__)

In [ ]:
# To ensure reproducibility of results
from qiskit_machine_learning.utils import algorithm_globals
algorithm_globals.random_seed = 12345

In [ ]:
# --- Import Libraries ---
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score, recall_score, balanced_accuracy_score
from scipy.stats import chi2_contingency

In [ ]:
# --- Qiskit Imports ---
from qiskit.circuit.library import ZZFeatureMap
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit_machine_learning.kernels import FidelityQuantumKernel

##### Load and Preprocess Dataset

In [ ]:
# --- Load Lung Cancer Dataset ---
lung_cancer_column_names = ['label'] + [f'attr_{i}' for i in range(1, 57)]
file_path = r'C:\Users\User\Documents\MyProjects\FYP_ResearchProject\data\lung+cancer\lung-cancer.data'

# Read data, treating "?" as missing values
df = pd.read_csv(file_path, header=None, names=lung_cancer_column_names, na_values=['?'])

print(f"Original shape of Lung Cancer data: {df.shape}")

# Mode imputation for missing values
modes = df.mode().iloc[0]
df.fillna(modes, inplace=True)
print(f"Total missing values after imputation: {df.isnull().sum().sum()}")

# Binary label: Class 1 -> 0, Others -> 1
df['label_binary'] = df['label'].apply(lambda x: 0 if x == 1 else 1)

print(f"\nClass distribution:")
print(df['label_binary'].value_counts())

In [ ]:
# --- Feature Selection Helper: Cramér's V ---
def cramers_v(x, y):
    """Calculate Cramér's V statistic for categorical-categorical association."""
    confusion_matrix = pd.crosstab(x, y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2 / n
    r, k = confusion_matrix.shape
    phi2corr = max(0, phi2 - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    if min((kcorr-1), (rcorr-1)) == 0:
        return 0
    return np.sqrt(phi2corr / min((kcorr-1), (rcorr-1)))

print("Cramér's V function ready for feature selection.")

##### Noise Model and Error Mitigation Functions

In [ ]:
# Base error rates (realistic NISQ device)
NOISE_CONFIGS = {
    'low': {'p_1q': 0.0001, 'p_2q': 0.001, 'p_ro': 0.002},
    'standard': {'p_1q': 0.001,  'p_2q': 0.01,  'p_ro': 0.02},
    'high': {'p_1q': 0.005,  'p_2q': 0.05,  'p_ro': 0.10}
}

def get_scaled_noise_model(scale_factor=1.0, level='standard'):
    """
    Build a noise model with scaled error probabilities for ZNE.
    Uses formula: p_scaled = 1 - (1-p)^scale_factor
    """
    config = NOISE_CONFIGS.get(level, NOISE_CONFIGS['standard'])
    
    p_1q = config['p_1q']
    p_2q = config['p_2q']
    p_ro = config['p_ro']

    p_1q_scaled = 1 - (1 - p_1q)**scale_factor
    p_2q_scaled = 1 - (1 - p_2q)**scale_factor
    p_ro_scaled = 1 - (1 - p_ro)**scale_factor
    
    noise_model = NoiseModel()
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_1q_scaled, 1), ['u1', 'u2', 'u3'])
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p_2q_scaled, 2), ['cx'])
    readout_error = ReadoutError([[1 - p_ro_scaled, p_ro_scaled], [p_ro_scaled, 1 - p_ro_scaled]])
    noise_model.add_all_qubit_readout_error(readout_error)
    
    backend = AerSimulator(noise_model=noise_model, seed_simulator=12345)
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    
    return noise_model, backend, pm, config

print("Noise model factory function ready!")

In [ ]:
# ==========================================
# READOUT ERROR MITIGATION (REM) FUNCTION
# ==========================================

def apply_rem_to_kernel(kernel_matrix, p_ro, n_qubits):
    """
    Apply Readout Error Mitigation to a kernel matrix.
    
    The correction formula for fidelity with symmetric readout error:
    K_corrected = (K_noisy - bias) / correction_factor
    """
    correction_factor = (1 - 2 * p_ro) ** n_qubits
    
    if abs(correction_factor) < 1e-10:
        print("Warning: Correction factor too small, using raw kernel")
        return kernel_matrix
    
    bias = 0.5 * (1 - correction_factor)
    corrected_kernel = (kernel_matrix - bias) / correction_factor
    
    # Clip to valid kernel range [0, 1]
    corrected_kernel = np.clip(corrected_kernel, 0, 1)
    
    # Ensure diagonal is exactly 1 (self-similarity)
    if kernel_matrix.shape[0] == kernel_matrix.shape[1]:
        np.fill_diagonal(corrected_kernel, 1.0)
    
    return corrected_kernel

print("REM function ready!")

##### Experiment Configurations (ZNE + REM Combined Only)

In [ ]:
# ==========================================
# EXPERIMENT CONFIGURATIONS
# ==========================================
#
# Focus: Combined ZNE+REM for all experiments.
# Feature selection uses Cramér's V (Lung Cancer is categorical).
# No sample size variation due to small dataset (32 samples).
#
# ==========================================

experiments = [
    # --- EXP 1: Dimensionality Effect ---
    {'id': 'Exp1_4feat',   'k_features': 4,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_6feat',   'k_features': 6,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_8feat',   'k_features': 8,  'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp1_10feat',  'k_features': 10, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 2: Shot Noise Effect ---
    {'id': 'Exp2_512shots',  'k_features': 8, 'shots': 512,  'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_1024shots', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp2_2048shots', 'k_features': 8, 'shots': 2048, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 3: Reps Effect ---
    {'id': 'Exp3_Reps1', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_Reps2', 'k_features': 8, 'shots': 1024, 'reps': 2, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp3_Reps3', 'k_features': 8, 'shots': 1024, 'reps': 3, 'entanglement': 'linear', 'noise_level': 'standard'},

    # --- EXP 4: Entanglement Effect ---
    {'id': 'Exp4_Linear',   'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear',   'noise_level': 'standard'},
    {'id': 'Exp4_Circular', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'circular', 'noise_level': 'standard'},
    {'id': 'Exp4_Full',     'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'full',     'noise_level': 'standard'},

    # --- EXP 5: Noise Level Effect ---
    {'id': 'Exp5_LowNoise',  'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'low'},
    {'id': 'Exp5_StdNoise',  'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'standard'},
    {'id': 'Exp5_HighNoise', 'k_features': 8, 'shots': 1024, 'reps': 1, 'entanglement': 'linear', 'noise_level': 'high'},
]

print(f"Total experiments configured: {len(experiments)}")
print("All experiments use combined ZNE+REM error mitigation")

##### Main Experiment Loop (ZNE+REM Combined)

In [ ]:
from qiskit_machine_learning.utils import algorithm_globals

# ZNE scales for Richardson extrapolation
ZNE_SCALES = [1.0, 3.0]

all_results = []

for i, config in enumerate(experiments, 1):
    print("="*80)
    print(f"EXPERIMENT {i}/{len(experiments)}: {config['id']} (ZNE+REM Mitigated)")
    print("="*80)
    print(f"  K Features: {config['k_features']} | Shots: {config['shots']}")
    print(f"  Noise Level: {config['noise_level']} | Reps: {config['reps']} | Entanglement: {config['entanglement']}")
    
    start_time = time.time()
    
    # --- 1. Data Preparation ---
    X = df.drop(['label', 'label_binary'], axis=1)
    y = df['label_binary']
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.30, random_state=42, stratify=y
    )
    
    # One-hot encoding
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train), columns=encoder.get_feature_names_out())
    X_test_encoded = pd.DataFrame(encoder.transform(X_test), columns=encoder.get_feature_names_out())
    
    # Feature Selection (Cramér's V)
    k = config['k_features']
    cramers_scores = {col: cramers_v(X_train_encoded[col], y_train) for col in X_train_encoded.columns}
    cramers_series = pd.Series(cramers_scores).sort_values(ascending=False)
    top_features = cramers_series.head(k).index.tolist()
    
    X_train_kbest = X_train_encoded[top_features].values
    X_test_kbest = X_test_encoded[top_features].values
    
    # Get noise config for REM
    noise_config = NOISE_CONFIGS.get(config['noise_level'], NOISE_CONFIGS['standard'])
    p_ro = noise_config['p_ro']
    n_qubits = k
    
    # --- 2. Compute Kernels (ZNE Scales) ---
    kernels_train = {}
    kernels_test = {}
    
    feature_map = ZZFeatureMap(
        feature_dimension=k, 
        reps=config['reps'], 
        entanglement=config['entanglement']
    )
    
    print(f"  Computing quantum kernels...")
    start_kernel = time.time()
    for scale in ZNE_SCALES:
        print(f"  → Scale factor {scale}...", end=" ", flush=True)
        _, backend, pm, _ = get_scaled_noise_model(scale_factor=scale, level=config['noise_level'])
        sampler = AerSampler.from_backend(backend=backend, default_shots=config['shots'])
        fidelity = ComputeUncompute(sampler=sampler, pass_manager=pm)
        qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=feature_map)
        
        kernels_train[scale] = qkernel.evaluate(x_vec=X_train_kbest)
        kernels_test[scale] = qkernel.evaluate(x_vec=X_test_kbest, y_vec=X_train_kbest)
        print("Done")

    kernel_time = time.time() - start_kernel
    print(f"  → Total kernel time: {kernel_time:.2f}s")
    
    # --- 3. Apply ZNE + REM ---
    print(f"  Applying error mitigation...")
    # Step 1: ZNE (Linear Richardson)
    kernel_train_zne = 1.5 * kernels_train[1.0] - 0.5 * kernels_train[3.0]
    kernel_test_zne = 1.5 * kernels_test[1.0] - 0.5 * kernels_test[3.0]
    
    # Step 2: REM
    kernel_train_final = apply_rem_to_kernel(kernel_train_zne, p_ro, n_qubits)
    kernel_test_final = apply_rem_to_kernel(kernel_test_zne, p_ro, n_qubits)
    print(f"  → ZNE+REM applied (p_ro={p_ro}, n_qubits={n_qubits})")
    
    # Ensure valid kernel values
    kernel_train_final = np.clip(kernel_train_final, 0, 1)
    kernel_test_final = np.clip(kernel_test_final, 0, 1)
    
    # --- 4. Train SVC ---
    print(f"  Grid searching for optimal C...")
    param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    svc = SVC(kernel='precomputed', class_weight='balanced')
    grid = GridSearchCV(svc, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
    
    start_train = time.time()
    grid.fit(kernel_train_final, y_train)
    train_time = time.time() - start_train
    
    best_model = grid.best_estimator_
    best_c = grid.best_params_['C']
    cv_score = grid.best_score_
    
    # Predictions
    y_train_pred = best_model.predict(kernel_train_final)
    y_test_pred = best_model.predict(kernel_test_final)
    
    # Metrics
    train_acc = accuracy_score(y_train, y_train_pred)
    test_acc = accuracy_score(y_test, y_test_pred)
    test_balanced_acc = balanced_accuracy_score(y_test, y_test_pred)
    recall = recall_score(y_test, y_test_pred, pos_label=1)
    gen_gap = abs(train_acc - test_acc)
    
    elapsed_time = time.time() - start_time
    
    print(f"  → Best C: {best_c}")
    print(f"  → Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")
    print(f"  → Recall: {recall:.4f} | Gen Gap: {gen_gap:.4f}")
    
    # Store results (Standardized keys)
    all_results.append({
        'experiment_id': config['id'],
        'k_features': config['k_features'],
        'shots': config['shots'],
        'reps': config['reps'],
        'entanglement': config['entanglement'],
        'noise_level': config['noise_level'],
        'best_c': best_c,
        'cv_score': cv_score,
        'train_acc': train_acc,
        'test_acc': test_acc,
        'test_balanced_acc': test_balanced_acc,
        'spam_recall': recall,
        'gen_gap': gen_gap,
        'kernel_time': kernel_time,
        'train_time': train_time,
        'total_time': elapsed_time
    })
    
    # --- SAVE KERNEL MATRICES ---
    np.save(f'kernel_train_{config["id"]}_lung.npy', kernel_train_final)
    np.save(f'kernel_test_{config["id"]}_lung.npy', kernel_test_final)
    print(f"  → Saved kernels to .npy files")
    
    print(f"\nExperiment {i}/{len(experiments)} complete!")

# Save Results CSV
results_df = pd.DataFrame(all_results)
results_df.to_csv('em_qsvm_lungcancer_znerem_results.csv', index=False)
print("\n" + "="*80)
print("ALL EXPERIMENTS COMPLETE!")
print("Results saved to: em_qsvm_lungcancer_znerem_results.csv")
print("="*80)

##### Results Summary

In [ ]:
# Display summary table
print("\nResults Summary:")
print(results_df[['experiment_id', 'k_features', 'noise_level', 'test_acc', 'spam_recall', 'gen_gap']].to_string(index=False))

In [ ]:
# ==========================================
# VISUALIZATION
# ==========================================

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')

def plot_experiment_group(df, group_prefix, param_col, xlabel):
    subset = df[df['experiment_id'].str.contains(group_prefix)].copy()
    if subset.empty:
        return
    
    subset = subset.sort_values(param_col)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Performance
    axes[0].plot(subset[param_col], subset['test_acc'], 'o-', label='Test Accuracy')
    axes[0].plot(subset[param_col], subset['spam_recall'], 's--', label='Recall')
    axes[0].set_xlabel(xlabel)
    axes[0].set_ylabel('Score')
    axes[0].set_title(f'Performance vs {xlabel}')
    axes[0].legend()
    
    # Gen Gap
    axes[1].plot(subset[param_col], subset['gen_gap'], 'D-', color='red')
    axes[1].set_xlabel(xlabel)
    axes[1].set_ylabel('Gap')
    axes[1].set_title('Generalization Gap')
    
    plt.tight_layout()
    plt.show()

plot_experiment_group(results_df, 'Exp1', 'k_features', 'Features (Qubits)')
plot_experiment_group(results_df, 'Exp2', 'shots', 'Shots')

In [ ]:
# Heatmap
plt.figure(figsize=(12, 8))
heatmap_data = results_df.set_index('experiment_id')[['test_acc', 'spam_recall', 'gen_gap', 'train_acc']]
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn', center=0.75)
plt.title('ZNE+REM Error Mitigated QSVM - Lung Cancer')
plt.tight_layout()
plt.savefig('lungcancer_em_heatmap.png', dpi=150)
plt.show()